# YouTube → Separation → Zero-Contamination Experiment

This notebook mirrors the SonicStudio **Experiment** funnel, with one processing step per cell. Run it on the model server (`vsf-242`) with the repository `.venv` kernel. Downloads, stems, previews, and saved diarization results stay under the repository `.data/` directory.

- **Flexible Model Selection:** Choose your source separation backend (`MelRoFormer`, `BSRoFormer`, `HTDemucs`, `MVSepMDX23`, or bypass separation), primary and secondary diarization engines (`Sortformer`, `DiariZen`, `Pyannote Community 1`, `Pyannote 3.1`), syllable aligner (`PhoWhisper`, `Whisper`, `MMS-FA`), and foundation verifiers (`Gemini Flash`, `Gemma 4`, `VibeVoice-ASR`).
- **Newest Zero-Contamination Config:** Includes Stage 3d Intelligent ASR & Pause-Guided Turn Segmentation (TTS sentence sizing), competitor tripwires, context-aware collars, and gate bypass safeguards.

In [ ]:
from __future__ import annotations

import os
import sys
from dataclasses import replace
from html import escape as html_escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio as IPythonAudio, HTML, clear_output, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf

# Works when Jupyter starts in src/notebooks/ as documented, and also from repo subdirectories.
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Could not find the repository root (pyproject.toml).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

from src.data_paths import DATA_DIR
from src.utils.AudioClass import Audio
from src.utils.AudioCutter import AudioCutter
from src.yt_crawler.YtCrawlerClass import YtCrawler

# Source Separation Backends
from src.separation import (
    BaseSeparator,
    BSRoFormer,
    HTDemucs,
    MelRoFormer,
    MVSepMDX23,
)

# Speaker Diarization Backends & Schema
from src.diarization import (
    BaseDiarizer,
    ClusteringWorkerDiarizer,
    DiariZenDiarizer,
    DiariZenWorkerDiarizer,
    DiarizationFilter,
    DiarizationModelInfo,
    DiarizationResult,
    clean_speaker_turns,
    PyannoteDiarizer,
    SortformerDiarizer,
    SortformerWorkerDiarizer,
    Speaker,
    SpeakerTurn,
    SpeakerVerifier,
    ThreeDSpeakerWorkerDiarizer,
    ZeroContaminationConfig,
    ZeroContaminationResult,
    DEFAULT_EMBEDDING_MODEL_ID,
)

# Pipeline Stages & Defaults
from src.diarization.zero_contamination import (
    DEFAULT_COLLAR_EROSION_S,
    DEFAULT_COMPETITOR_ONSET,
    DEFAULT_ENERGY_FRAME_LEN_MS,
    DEFAULT_ENERGY_HOP_LEN_MS,
    DEFAULT_ENERGY_SEARCH_WINDOW_S,
    DEFAULT_ENERGY_VALLEY_FLOOR_DB,
    DEFAULT_HANDOFF_RISK_DISTANCE_S,
    DEFAULT_HOMOGENEITY_HOP_S,
    DEFAULT_HOMOGENEITY_WINDOW_S,
    DEFAULT_MIN_HOMOGENEITY_SIMILARITY,
    DEFAULT_MIN_SPLIT_PAUSE_S,
    DEFAULT_MIN_TURN_DURATION_S,
    DEFAULT_SILENCE_TAIL_BUFFER_S,
    DEFAULT_TARGET_MAX_DURATION_S,
    DEFAULT_TARGET_MIN_DURATION_S,
    DEFAULT_TARGET_OFFSET,
    DEFAULT_TARGET_ONSET,
    DEFAULT_TRANSITION_EXCLUSION_S,
    align_and_lock_syllable_boundaries,
    apply_context_aware_collar,
    compute_consensus_turns,
    filter_by_embedding_homogeneity,
    filter_by_foundation_models,
    run_zero_contamination_pipeline,
    smart_segment_speaker_turns,
    snap_boundaries_to_acoustic_valleys,
)


## Input Audio & Device Allocation

Specify your input audio target and compute device. Each downstream stage has its own local parameters directly inside its cell for maximum interactive flexibility.

In [ ]:
# Input audio: specify YouTube URL or point to a local audio file
URL = "https://www.youtube.com/watch?v=REPLACE_ME"
LOCAL_AUDIO_PATH: Path | None = "../../.data/mel_roformer/out/Khám_phá_Top_Đại_Học_Siêu_Hot_Bách_Khoa,_Khoa_Học_Tự_Nhiên,_Công_Nghệ_VyUni_Ep.2_khối_KH-KT-CN__mEHOvkGFz54.wav"  # e.g. Path("/path/to/local.wav")

# Compute devices & credentials
DEVICE = "cuda:0"
HF_TOKEN = os.getenv("HF_TOKEN")

# Stage attrition tracking
stage_stats = {}
def record_stage(name: str, turns) -> list[SpeakerTurn]:
    turns = list(turns)
    stage_stats[name] = {
        "turns": len(turns),
        "speech_duration_s": round(sum(turn.duration_s for turn in turns), 2),
    }
    display(stage_stats[name])
    return turns


## Model Initializers — Flexible Backend Selection

The factories below expose explicit initialization for every separator and diarizer backend. You can either use these functions or directly instantiate your preferred model class.

In [ ]:
def init_separator(
    backend: str = "mel_roformer",
    *,
    device: str = DEVICE,
    output_dir: Path | str | None = None,
    work_dir: Path | str | None = None,
    model_name: str | None = None,
    two_stems: str = "vocals",
    **kwargs,
) -> BaseSeparator | None:
    """Initialize any supported source separator, or return None for vocal passthrough.

    Supported backends:
      - 'mel_roformer': MelRoFormer (mel_band_roformer_vocals_fv8pndlq)
      - 'bs_roformer': BSRoFormer (bs_roformer_ep_317_sdr_12.9755.ckpt)
      - 'htdemucs': HTDemucs (htdemucs_ft)
      - 'mvsep': MVSepMDX23
      - 'none' / None: Passthrough (skip separation if audio is already clean speech/vocal)
    """
    b = (backend or "").lower().strip()
    if not b or b in {"none", "passthrough"}:
        return None
    out_d = Path(output_dir) if output_dir else DATA_DIR / "notebook" / "zero_contamination" / "stems"
    wk_d = Path(work_dir) if work_dir else DATA_DIR / "notebook" / "zero_contamination" / "separation_work"

    if b in {"mel_roformer", "melroformer", "mel"}:
        return MelRoFormer(
            model_name=model_name or "mel_band_roformer_vocals_fv8pndlq",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"bs_roformer", "bsroformer", "bs"}:
        return BSRoFormer(
            model_name=model_name or "bs_roformer_ep_317_sdr_12.9755.ckpt",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"htdemucs", "demucs"}:
        return HTDemucs(
            model_name=model_name or "htdemucs_ft",
            device=device,
            two_stems=two_stems,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    elif b in {"mvsep", "mvsepmdx23", "mdx23"}:
        return MVSepMDX23(
            device=device,
            output_dir=out_d,
            work_dir=wk_d,
            **kwargs,
        )
    else:
        raise ValueError(
            f"Unsupported separator backend: {backend}. "
            f"Choose from 'mel_roformer', 'bs_roformer', 'htdemucs', 'mvsep', or 'none'."
        )


def init_diarizer(
    backend: str,
    *,
    device: str = "auto",
    token: str | None = None,
    onset: float = DEFAULT_TARGET_ONSET,
    offset: float = DEFAULT_TARGET_OFFSET,
    model_id: str | None = None,
    isolated: bool = True,
    **kwargs,
) -> BaseDiarizer:
    """Initialize any supported speaker diarizer backend for primary or secondary stages.

    Supported backends:
      - 'sortformer': SortformerWorkerDiarizer (isolated subprocess) or SortformerDiarizer
      - 'diarizen': DiariZenWorkerDiarizer (isolated subprocess) or DiariZenDiarizer
      - 'pyannote' / 'pyannote_community': PyannoteDiarizer (community-1)
      - 'pyannote_31': PyannoteDiarizer (v3.1)
      - 'clustering': ClusteringWorkerDiarizer
      - '3dspeaker': ThreeDSpeakerWorkerDiarizer
    """
    b = backend.lower().strip()
    tok = token or os.getenv("HF_TOKEN")
    if b in {"sortformer", "nemo-sortformer"}:
        if isolated:
            return SortformerWorkerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
        return SortformerDiarizer(device=device, token=tok, onset=onset, offset=offset, **kwargs)
    elif b in {"diarizen", "diarizen_large_s80_v2"}:
        if isolated:
            return DiariZenWorkerDiarizer(device=device, token=tok, **kwargs)
        return DiariZenDiarizer(device=device, token=tok, **kwargs)
    elif b in {"pyannote", "pyannote_community"}:
        mid = model_id or "pyannote/speaker-diarization-community-1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"pyannote_31", "pyannote_3.1"}:
        mid = model_id or "pyannote/speaker-diarization-3.1"
        return PyannoteDiarizer(model_id=mid, device=device, token=tok, **kwargs)
    elif b in {"clustering", "clustering_worker"}:
        return ClusteringWorkerDiarizer(device=device, token=tok, **kwargs)
    elif b in {"3dspeaker", "3d_speaker"}:
        return ThreeDSpeakerWorkerDiarizer(device=device, token=tok, **kwargs)
    else:
        raise ValueError(
            f"Unsupported diarizer backend: {backend}. "
            f"Choose from 'sortformer', 'diarizen', 'pyannote', 'pyannote_31', 'clustering', '3dspeaker'."
        )


## Input — Load local audio or crawl YouTube URL

In [ ]:
if LOCAL_AUDIO_PATH is not None and Path(LOCAL_AUDIO_PATH).is_file():
    source_audio = Audio.from_file(LOCAL_AUDIO_PATH)
    print(f"Loaded local audio file: {source_audio.path}")
else:
    crawler = YtCrawler(
        output_dir=DATA_DIR / "notebook" / "zero_contamination" / "downloads",
        work_dir=DATA_DIR / "notebook" / "zero_contamination" / "crawl_work",
    )
    source_audio: Audio = crawler.download(URL)

display(source_audio.metadata())
source_audio.notebook_display()


## Separation — Vocals Stem Extraction

Select your separator (`mel_roformer`, `bs_roformer`, `htdemucs`, `mvsep`, or `'none'` for passthrough).

In [ ]:
# Change SEPARATION_BACKEND to 'mel_roformer', 'bs_roformer', 'htdemucs', 'mvsep', or 'none'
SEPARATION_BACKEND = "mel_roformer"

separator = init_separator(
    backend=SEPARATION_BACKEND,
    device=DEVICE,
    output_dir=DATA_DIR / "notebook" / "zero_contamination" / "stems",
    work_dir=DATA_DIR / "notebook" / "zero_contamination" / "separation_work",
)

if separator is not None:
    with separator:
        speech_audio: Audio = separator.separate(source_audio)
else:
    print("Separation skipped (passthrough mode) — using source audio directly.")
    speech_audio = source_audio

display(speech_audio.metadata())
speech_audio.notebook_display()


## Experiment Stage 1 — Primary Diarization

In [ ]:
# Stage 1 Parameters
PRIMARY_BACKEND = "sortformer"  # "sortformer", "diarizen", "pyannote", "pyannote_31"
TARGET_ONSET = 0.80
TARGET_OFFSET = 0.65

primary_diarizer = init_diarizer(
    backend=PRIMARY_BACKEND,
    device=DEVICE,
    token=HF_TOKEN,
    onset=TARGET_ONSET,
    offset=TARGET_OFFSET,
)

with primary_diarizer:
    primary_result: DiarizationResult = primary_diarizer.diarize(speech_audio)

current_turns = record_stage(
    "1_primary", sorted(primary_result.turns, key=lambda turn: turn.start_s)
)


## Experiment Stage 2 — Dual-Engine Mutual Hungarian Consensus

In [ ]:
# Stage 2 Parameters
ENABLE_CONSENSUS = True
SECONDARY_BACKEND = "diarizen"  # "diarizen", "sortformer", "pyannote", "pyannote_31"
SECONDARY_DEVICE = DEVICE       # "cuda:0", "cuda:1", or "cpu"

if ENABLE_CONSENSUS:
    secondary_diarizer = init_diarizer(
        backend=SECONDARY_BACKEND,
        device=SECONDARY_DEVICE,
        token=HF_TOKEN,
    )
    with secondary_diarizer:
        secondary_result: DiarizationResult = secondary_diarizer.diarize(speech_audio)
    current_turns, speaker_mapping = compute_consensus_turns(
        current_turns, secondary_result.turns, speech_audio.duration_s
    )
    current_turns = record_stage("2_consensus", current_turns)
    display({"speaker_mapping": speaker_mapping})
else:
    print("Stage 2 (Consensus) skipped.")


## Experiment Stage 3a — Context-Aware Collar & Handoff Guard

In [ ]:
# Stage 3a Parameters
ENABLE_CONTEXT_COLLAR = True
BOUNDARY_COLLAR_S = 0.35
HANDOFF_RISK_S = 0.80
SILENCE_TAIL_S = 0.027
MIN_TURN_DURATION_S = 0.80
TRANSITION_EXCLUSION_S = 0.50

if ENABLE_CONTEXT_COLLAR:
    current_turns, collar_audits = apply_context_aware_collar(
        current_turns,
        collar_s=BOUNDARY_COLLAR_S,
        handoff_risk_s=HANDOFF_RISK_S,
        silence_tail_s=SILENCE_TAIL_S,
        min_duration_s=MIN_TURN_DURATION_S,
        transition_exclusion_s=TRANSITION_EXCLUSION_S,
        audio_duration_s=speech_audio.duration_s,
    )
    current_turns = record_stage("3a_context_collar", current_turns)
else:
    print("Stage 3a (Context collar) skipped.")


## Experiment Stage 3b — Syllable / Word Forced-Alignment Lock

In [ ]:
# Stage 3b Parameters
ENABLE_SYLLABLE_ALIGNMENT = True
ALIGNER_ENGINE = "whisper_timestamped"  # "whisper_timestamped", "mms_fa", "remote_whisper"
ALIGNER_MODEL = "vinai/PhoWhisper-small"  # "vinai/PhoWhisper-small", "vinai/PhoWhisper-large"
ALIGNER_LANGUAGE = "vi"
ALIGNER_DEVICE = "cpu"                  # "cpu" recommended to avoid VRAM contention
ALIGNER_ENDPOINT = os.getenv("WHISPER_ENDPOINT")

if ENABLE_SYLLABLE_ALIGNMENT:
    current_turns, alignment_audits = align_and_lock_syllable_boundaries(
        speech_audio,
        current_turns,
        aligner_engine=ALIGNER_ENGINE,
        aligner_model=ALIGNER_MODEL,
        aligner_language=ALIGNER_LANGUAGE,
        aligner_endpoint=ALIGNER_ENDPOINT,
        aligner_device=ALIGNER_DEVICE,
        token=HF_TOKEN,
    )
    current_turns = record_stage("3b_word_lock", current_turns)
else:
    print("Stage 3b (Syllable alignment) skipped.")


## Experiment Stage 3c — Micro-Energy Valley Snapping

In [ ]:
# Stage 3c Parameters
ENABLE_ENERGY_SNAPPING = True
ENERGY_SEARCH_WINDOW_S = 0.15
ENERGY_VALLEY_FLOOR_DB = -30.0
ENERGY_FRAME_LEN_MS = 2.0
ENERGY_HOP_LEN_MS = 0.5

if ENABLE_ENERGY_SNAPPING:
    current_turns, energy_audits = snap_boundaries_to_acoustic_valleys(
        speech_audio,
        current_turns,
        search_window_s=ENERGY_SEARCH_WINDOW_S,
        energy_floor_db=ENERGY_VALLEY_FLOOR_DB,
        frame_len_ms=ENERGY_FRAME_LEN_MS,
        hop_len_ms=ENERGY_HOP_LEN_MS,
    )
    current_turns = record_stage("3c_energy_snap", current_turns)
else:
    print("Stage 3c (Energy snapping) skipped.")


## Experiment Stage 3d — Intelligent ASR & Pause-Guided Turn Segmentation (TTS Sentence Sizing)

Segments long turns into optimal TTS training slices (3–10s) using ASR punctuation and breathing pauses, snapping cut points to local acoustic energy valleys.

In [ ]:
# Stage 3d Parameters
ENABLE_SMART_SEGMENTATION = True
TARGET_MAX_DURATION_S = 10.0
TARGET_MIN_DURATION_S = 3.0
MIN_SPLIT_PAUSE_S = 0.20

if ENABLE_SMART_SEGMENTATION:
    current_turns, segment_audits = smart_segment_speaker_turns(
        speech_audio,
        current_turns,
        max_duration_s=TARGET_MAX_DURATION_S,
        min_duration_s=TARGET_MIN_DURATION_S,
        min_pause_s=MIN_SPLIT_PAUSE_S,
        search_window_s=ENERGY_SEARCH_WINDOW_S,
        frame_len_ms=ENERGY_FRAME_LEN_MS,
        hop_len_ms=ENERGY_HOP_LEN_MS,
    )
    current_turns = record_stage("3d_smart_segmentation", current_turns)
else:
    print("Stage 3d (Smart segmentation) skipped.")


## Experiment Stage 4 — WeSpeaker Sliding-Window Homogeneity

In [ ]:
# Stage 4 Parameters
ENABLE_HOMOGENEITY = True
HOMOGENEITY_DEVICE = DEVICE     # "cuda:0", "cuda:1", or "cpu"
HOMOGENEITY_WINDOW_S = 1.00
HOMOGENEITY_HOP_S = 0.25
MIN_HOMOGENEITY_SIMILARITY = 0.75

if ENABLE_HOMOGENEITY:
    current_turns, homogeneity_audits = filter_by_embedding_homogeneity(
        speech_audio,
        current_turns,
        window_s=HOMOGENEITY_WINDOW_S,
        hop_s=HOMOGENEITY_HOP_S,
        min_similarity=MIN_HOMOGENEITY_SIMILARITY,
        device=HOMOGENEITY_DEVICE,
        token=HF_TOKEN,
    )
    current_turns = record_stage("4_homogeneity", current_turns)
else:
    print("Stage 4 (Homogeneity) skipped.")


## Experiment Stage 5a — Gemma/Gemini Direct-Audio Verifier

Verifies acoustic speaker purity and word completeness (không bị lẹm chữ).

In [ ]:
# Stage 5a Parameters
ENABLE_DIRECT_AUDIO = True
DIRECT_AUDIO_BACKEND = "gemini"         # "gemini" or "gemma4"
DIRECT_AUDIO_MODEL = "gemini-3.8-flash"  # "gemini-3.8-flash", "gemini-3.1-flash-lite", "unsloth/gemma-4-12b-it-GGUF"
DIRECT_AUDIO_CONCURRENCY = 10           # Parallel requests (Gemini: 10, Gemma 4: 1)
DIRECT_AUDIO_TIMEOUT_S = 120.0
DIRECT_AUDIO_MAX_TOKENS = 1024
DIRECT_AUDIO_ENDPOINT = os.getenv("UNSLOTH_ENDPOINT")  # only needed for local gemma4

if ENABLE_DIRECT_AUDIO:
    current_turns, direct_audio_audits = filter_by_foundation_models(
        speech_audio,
        current_turns,
        enable_gemma=True,
        enable_vibevoice=False,
        gemma_backend=DIRECT_AUDIO_BACKEND,
        gemma_model=DIRECT_AUDIO_MODEL,
        gemma_endpoint=DIRECT_AUDIO_ENDPOINT,
        gemma_concurrency=DIRECT_AUDIO_CONCURRENCY,
        gemma_timeout_s=DIRECT_AUDIO_TIMEOUT_S,
        gemma_max_output_tokens=DIRECT_AUDIO_MAX_TOKENS,
    )
    current_turns = record_stage("5a_direct_audio", current_turns)
else:
    print("Stage 5a (Direct audio verifier) skipped.")


## Experiment Stage 5b — VibeVoice-ASR Speaker-Count Verifier

In [ ]:
# Stage 5b Parameters
ENABLE_VIBEVOICE = True
VIBEVOICE_MODEL_ID = "Dubedo/VibeVoice-ASR-HF-INT8"
VIBEVOICE_DEVICE = "cuda:1"              # dedicated secondary GPU recommended
VIBEVOICE_ENDPOINT = os.getenv("VIBEVOICE_ENDPOINT")
MAX_SECONDARY_SPEECH_S = 0.0

if ENABLE_VIBEVOICE:
    current_turns, vibevoice_audits = filter_by_foundation_models(
        speech_audio,
        current_turns,
        enable_gemma=False,
        enable_vibevoice=True,
        vibevoice_model_id=VIBEVOICE_MODEL_ID,
        vibevoice_device=VIBEVOICE_DEVICE,
        vibevoice_endpoint=VIBEVOICE_ENDPOINT,
        max_secondary_speech_s=MAX_SECONDARY_SPEECH_S,
    )
    current_turns = record_stage("5b_vibevoice", current_turns)
else:
    print("Stage 5b (VibeVoice verifier) skipped.")


## Assemble and Persist Canonical Diarization Result

In [ ]:
speaker_ids = sorted({turn.speaker_id for turn in current_turns})
consensus_tag = f"+{SECONDARY_BACKEND}" if ENABLE_CONSENSUS else ""
model_label = f"{PRIMARY_BACKEND}{consensus_tag}+notebook-gates"

diarization_result = DiarizationResult(
    schema_version="2.0",
    audio_id=speech_audio.source_id,
    speakers=[Speaker(speaker_id=speaker_id) for speaker_id in speaker_ids],
    turns=current_turns,
    source_audio=speech_audio,
    model=DiarizationModelInfo(
        backend="zero-contamination-notebook",
        model_id=model_label,
    ),
)
result_path = diarization_result.save(
    DATA_DIR / "notebook" / "zero_contamination" / "results"
)
display({"saved_to": str(result_path), "funnel": stage_stats})


## Diarization Result Filtering & Turn Cleanup (SonicStudio Diarization Controls)

Fine-grained post-processing tools matching SonicStudio Diarization tab:
- **`DiarizationFilter`**: Configurable filter criteria (speaker whitelist/blacklist, duration bounds, overlap exclusion/isolation, confidence thresholding, time bounds, custom predicates).
- **Turn Cleanup (`clean_speaker_turns`)**: A-B-A jitter relabeling, boundary collars between speakers, same-speaker gap merging, and short residual turn trimming.
- **Fluent Methods on `DiarizationResult`**: `result.filter(...)`, `result.clean(...)`, `result.for_speaker(spk_id)`, and `result.filter_with(diar_filter)` return new immutable result objects.


In [ ]:
# Filter Criteria (mirrors SonicStudio Diarization Tab filters)
FILTER_TARGET_SPEAKER: str | None = None       # e.g., "SPEAKER_00" to isolate one speaker
EXCLUDE_SPEAKERS: list[str] | None = None       # e.g., ["SPEAKER_01"]
MIN_TURN_DURATION_S: float | None = 1.0        # Exclude short turns (< 1.0s)
MAX_TURN_DURATION_S: float | None = None       # Exclude turns longer than this (seconds)
EXCLUDE_OVERLAPPING_SPEECH: bool = True        # Exclude turns overlapping another speaker

# Turn Cleanup Settings (A-B-A jitter correction, collars, and same-speaker gap merging)
ENABLE_TURN_CLEANUP: bool = True
CLEANUP_MIN_DURATION_S: float = 0.5
CLEANUP_MERGE_GAP_S: float = 1.0
CLEANUP_BOUNDARY_COLLAR_S: float = 0.04
CLEANUP_JITTER_MAX_DURATION_S: float = 3.0

# Method 1: Using DiarizationFilter
diar_filter = DiarizationFilter(
    speakers=FILTER_TARGET_SPEAKER,
    exclude_speakers=EXCLUDE_SPEAKERS,
    min_duration_s=MIN_TURN_DURATION_S,
    max_duration_s=MAX_TURN_DURATION_S,
    exclude_overlap=EXCLUDE_OVERLAPPING_SPEECH,
    clean_turns=ENABLE_TURN_CLEANUP,
    min_turn_duration_s=CLEANUP_MIN_DURATION_S,
    merge_same_speaker_gap_s=CLEANUP_MERGE_GAP_S,
    boundary_collar_s=CLEANUP_BOUNDARY_COLLAR_S,
    jitter_max_duration_s=CLEANUP_JITTER_MAX_DURATION_S,
)
filtered_diarization_result = diarization_result.filter_with(diar_filter)

# Method 2: Fluent chaining directly on DiarizationResult
# filtered_diarization_result = (
#     diarization_result
#     .clean(boundary_collar_s=0.04, merge_same_speaker_gap_s=1.0)
#     .filter(speakers="SPEAKER_00", min_duration_s=1.0, exclude_overlap=True)
# )

print(f"Original turns: {diarization_result.turn_count} ({diarization_result.speaker_count} speakers, {diarization_result.total_speech_duration_s:.1f}s total)")
print(f"Filtered turns: {filtered_diarization_result.turn_count} ({filtered_diarization_result.speaker_count} speakers, {filtered_diarization_result.total_speech_duration_s:.1f}s total)")
display(filtered_diarization_result.to_dict()["summary"])


## Diarization result notebook viewer

The viewer brings the Experiment result table's speaker filtering, transcript search, raw/refined boundary comparison, waveform context, and lazy per-turn audio playback into Jupyter.

In [ ]:
class DiarizationResultNotebookViewer:
    """Interactive Jupyter viewer for a file-backed ``DiarizationResult``.

    Turn clips are created lazily under ``.data/notebook/diarization_viewer``.
    The controls mirror SonicStudio's result viewer: filter by speaker or
    text, inspect boundary metadata, compare blunt/refined audio, and view
    the waveform around the selected turn.
    """

    def __init__(
        self,
        result: DiarizationResult,
        output_dir: str | Path = DATA_DIR / "notebook" / "diarization_viewer",
    ) -> None:
        if result.source_audio is None:
            raise ValueError("The diarization result has no source_audio.")
        if not Path(result.source_audio.path).is_file():
            raise FileNotFoundError(result.source_audio.path)
        self.result = result
        self.audio = result.source_audio
        self.output_dir = Path(output_dir) / result.result_id
        self.cutter = AudioCutter(output_dir=self.output_dir)
        self._filtered_indices: list[int] = []

        speakers = ["All speakers", *sorted({t.speaker_id for t in result.turns})]
        self.speaker = widgets.Dropdown(options=speakers, description="Speaker:")
        self.search = widgets.Text(description="Search:", placeholder="transcript or policy")
        self.turn = widgets.Dropdown(options=[], description="Turn:", layout=widgets.Layout(width="100%"))
        self.summary = widgets.HTML()
        self.detail = widgets.Output()

        self.speaker.observe(self._filters_changed, names="value")
        self.search.observe(self._filters_changed, names="value")
        self.turn.observe(self._turn_changed, names="value")
        self._refresh_filters()

    @staticmethod
    def _transcript(turn) -> str:
        return str(getattr(turn, "_transcript", getattr(turn, "transcript", "")) or "")

    @staticmethod
    def _policy(turn) -> str:
        return str(getattr(turn, "_boundary_policy", "standard"))

    def _filters_changed(self, _change=None) -> None:
        self._refresh_filters()

    def _refresh_filters(self) -> None:
        wanted_speaker = self.speaker.value
        query = self.search.value.strip().lower()
        matches = []
        for index, turn in enumerate(self.result.turns):
            if wanted_speaker != "All speakers" and turn.speaker_id != wanted_speaker:
                continue
            haystack = f"{turn.speaker_id} {self._transcript(turn)} {self._policy(turn)}".lower()
            if query and query not in haystack:
                continue
            matches.append(index)
        self._filtered_indices = matches
        duration = sum(self.result.turns[index].duration_s for index in matches)
        average = duration / len(matches) if matches else 0.0
        self.summary.value = (
            f"<b>{len(matches)}</b> clean turns &nbsp;•&nbsp; "
            f"<b>{duration:.1f}s</b> speech &nbsp;•&nbsp; average <b>{average:.1f}s</b>"
        )
        options = [
            (f"#{index + 1} · {self.result.turns[index].speaker_id} · "
             f"{self.result.turns[index].start_s:.2f}–{self.result.turns[index].end_s:.2f}s", index)
            for index in matches
        ]
        self.turn.options = options
        if not options:
            with self.detail:
                clear_output(wait=True)
                display(HTML("<i>No turns match the active filters.</i>"))

    def _turn_changed(self, change) -> None:
        if change.get("new") is not None:
            self._render_turn(int(change["new"]))

    def _clip(self, index: int, start_s: float, end_s: float, kind: str) -> Audio:
        path = self.output_dir / f"turn_{index:06d}_{kind}.wav"
        if path.is_file():
            return Audio.from_file(path)
        return self.cutter.cut(self.audio, start_s, end_s, output_path=path)

    def _waveform(self, start_s: float, end_s: float, raw_start_s: float, raw_end_s: float) -> None:
        context_start = max(0.0, min(start_s, raw_start_s) - 0.20)
        context_end = min(self.audio.duration_s, max(end_s, raw_end_s) + 0.20)
        with sf.SoundFile(str(self.audio.path)) as source:
            sample_rate = int(source.samplerate)
            source.seek(int(context_start * sample_rate))
            waveform = source.read(int((context_end - context_start) * sample_rate), dtype="float32", always_2d=True)
        mono = waveform.mean(axis=1) if len(waveform) else np.zeros(1, dtype=np.float32)
        stride = max(1, len(mono) // 5000)
        times = context_start + np.arange(0, len(mono), stride) / sample_rate
        figure, axis = plt.subplots(figsize=(12, 2.4))
        axis.plot(times, mono[::stride], color="#64748b", linewidth=0.7)
        axis.axvspan(raw_start_s, raw_end_s, color="#f59e0b", alpha=0.18, label="Blunt/raw")
        axis.axvspan(start_s, end_s, color="#10b981", alpha=0.20, label="Refined")
        axis.set(xlabel="Time (s)", ylabel="Amplitude", xlim=(context_start, context_end))
        axis.legend(loc="upper right")
        figure.tight_layout()
        plt.show()
        plt.close(figure)

    def _render_turn(self, index: int) -> None:
        turn = self.result.turns[index]
        raw_start = float(getattr(turn, "_raw_start_s", turn.start_s))
        raw_end = float(getattr(turn, "_raw_end_s", turn.end_s))
        delta_end = float(getattr(turn, "_delta_end_ms", 0.0))
        transcript = self._transcript(turn) or "—"
        metadata = (
            "<table style='width:100%;text-align:left'>"
            f"<tr><th>Speaker</th><td>{html_escape(turn.speaker_id)}</td><th>Duration</th><td>{turn.duration_s:.2f}s</td></tr>"
            f"<tr><th>Refined</th><td>{turn.start_s:.3f}–{turn.end_s:.3f}s</td><th>Raw/blunt</th><td>{raw_start:.3f}–{raw_end:.3f}s</td></tr>"
            f"<tr><th>Policy</th><td>{html_escape(self._policy(turn))}</td><th>End delta</th><td>{delta_end:+.0f}ms</td></tr>"
            f"<tr><th>Transcript</th><td colspan='3'>{html_escape(transcript)}</td></tr></table>"
        )
        refined = self._clip(index, turn.start_s, turn.end_s, "refined")
        raw = self._clip(index, raw_start, raw_end, "raw")
        with self.detail:
            clear_output(wait=True)
            display(HTML(metadata))
            self._waveform(turn.start_s, turn.end_s, raw_start, raw_end)
            display(HTML("<b>Refined boundary</b>"))
            display(IPythonAudio(filename=str(refined.path)))
            if raw_start != turn.start_s or raw_end != turn.end_s:
                display(HTML("<b>Raw/blunt boundary</b>"))
                display(IPythonAudio(filename=str(raw.path)))

    def display(self) -> None:
        """Render the interactive viewer once and return ``None``."""
        controls = widgets.HBox([self.speaker, self.search])
        display(widgets.VBox([controls, self.summary, self.turn, self.detail]))
        if self.turn.value is not None:
            self._render_turn(int(self.turn.value))


viewer = DiarizationResultNotebookViewer(filtered_diarization_result)
viewer.display()
